# 09. Subsection concatenation

After each azimuth subsection is focused independently, `TomogramProcessor._concatenate` merges the partial HDF5 artifacts back into a single DEM and tomogram. DEM partials are stacked along axis 0 (azimuth) and tomogram partials along axis 1 (azimuth), with height as axis 0 and range as the trailing axis. This notebook writes synthetic partial HDF5 files, runs the real concatenation, and confirms the reassembled arrays match a known ground truth.

**Modules exercised:** pipelines.processing_pipeline.tomogram (TomogramProcessor._concatenate)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Build the real processor

We import the actual `TomogramProcessor` and a temporary directory layout that matches what `_concatenate` expects: partials under `TOMO/TOMO-SR`.

In [ ]:
import h5py
import tempfile
from configuration.processing_config import (
    ProcessingConfiguration, TomogramConfiguration, PathConfiguration,
)
from tools.data.regions import CropRegion
from pipelines.processing_pipeline.tomogram import TomogramProcessor
from tools.monitoring.logger import Logger

logger    = Logger(log_dir='/tmp/nb09_logs', name='nb09', level='WARNING')
config    = ProcessingConfiguration(crop=CropRegion(0, 400, 0, 100), paths=PathConfiguration(main_directory=Path('/tmp/nb09_run')))
processor = TomogramProcessor(config, logger)
print('processor ready')

## Define a ground-truth full tomogram and DEM

We choose the canonical shapes: DEM is `(azimuth, range)` and tomogram is `(height, azimuth, range)`. We then carve them into azimuth subsections that the workers would have produced.

In [ ]:
n_height, n_azimuth, n_range = 24, 400, 100

full_dem      = RNG.standard_normal((n_azimuth, n_range)).astype(np.float32)
full_tomogram = RNG.standard_normal((n_height, n_azimuth, n_range)).astype(np.float32)

boundaries = [0, 130, 260, 400]
azimuth_slices = [slice(boundaries[i], boundaries[i + 1]) for i in range(len(boundaries) - 1)]
print('subsection azimuth slices:', [(s.start, s.stop) for s in azimuth_slices])

## Write synthetic partial HDF5 files

Each partial holds the DEM slab and tomogram slab for its azimuth band, under the dataset names `DEM` and `tomogram` that `_concatenate` reads. The file names are zero-padded so `sorted(...)` restores azimuth order.

In [ ]:
temp_dir = Path(tempfile.mkdtemp(prefix='nb09_'))
partial_dir = temp_dir / 'TOMO' / 'TOMO-SR'
partial_dir.mkdir(parents=True, exist_ok=True)

for index, az_slice in enumerate(azimuth_slices):
    dem_slab  = full_dem[az_slice, :]
    tomo_slab = full_tomogram[:, az_slice, :]
    with h5py.File(partial_dir / f'part_{index:04d}.h5', 'w') as f:
        f.create_dataset('DEM', data=dem_slab)
        f.create_dataset('tomogram', data=tomo_slab)
print('wrote', len(azimuth_slices), 'partial files to', partial_dir)

## Run the real concatenation

`_concatenate` reads every partial, allocates the combined arrays, and copies each slab into place. We compare the result against the ground truth.

In [ ]:
combined_dem, combined_tomogram = processor._concatenate(temp_dir)

print('combined DEM shape      :', combined_dem.shape,      '(expected', full_dem.shape, ')')
print('combined tomogram shape :', combined_tomogram.shape, '(expected', full_tomogram.shape, ')')
print('DEM matches      :', np.allclose(combined_dem, full_dem))
print('tomogram matches :', np.allclose(combined_tomogram, full_tomogram))

## Visual confirmation

We plot the ground-truth DEM, the reassembled DEM, and their difference. The difference image should be uniformly zero, with the subsection boundaries (dashed lines) marking where partials were stitched.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
im0 = axes[0].imshow(full_dem, aspect='auto', origin='lower')
axes[0].set_title('ground-truth DEM')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
im1 = axes[1].imshow(combined_dem, aspect='auto', origin='lower')
axes[1].set_title('reassembled DEM')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
im2 = axes[2].imshow(combined_dem - full_dem, aspect='auto', origin='lower', cmap='bwr', vmin=-1e-6, vmax=1e-6)
axes[2].set_title('difference (should be 0)')
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
for b in boundaries[1:-1]:
    for ax in axes:
        ax.axhline(b, color='k', ls='--', lw=0.7)
for ax in axes:
    ax.set_xlabel('range'); ax.set_ylabel('azimuth')
fig.tight_layout()
plt.show()

In [ ]:
import shutil
shutil.rmtree(temp_dir, ignore_errors=True)
print('cleaned up', temp_dir)

## Expected visual outcome

The reassembled DEM should be pixel-identical to the ground truth, the difference panel uniformly zero, and both match-flags should report True. This confirms `_concatenate` stitches azimuth subsections back together in the correct order and along the correct axes.